# 03 — Data Preprocessing

## 1. Objective

The objective of this notebook is to prepare the UCI Cleveland Heart Disease dataset for Machine Learning model development while maintaining a reproducible and leakage-free preprocessing workflow.

The reusable preprocessing implementation is centralized in `src/preprocessing.py`. This notebook validates and demonstrates that shared workflow rather than redefining the preprocessing components locally. It will:

- Load and validate the Cleveland dataset using the shared module.
- Separate the 13 predictor variables from the original target variable.
- Convert `num` from the original 0–4 representation into the project binary target.
- Obtain the authoritative numerical and categorical feature groups from the shared module.
- Split the dataset using an 80:20 stratified split with `random_state=42`.
- Create the shared preprocessing transformer using `create_preprocessor()`.
- Fit the transformer on training data only for verification.
- Verify the transformed training and testing data and the resulting feature count.
- Confirm that the same preprocessing design can be embedded in model pipelines using `create_model_pipeline()`.

The resulting implementation can be reused consistently by the baseline, advanced-model, and future model-comparison notebooks without duplicating preprocessing logic.


## 2. Import Required Libraries

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    load_data,
    validate_columns,
    prepare_features_target,
    create_preprocessor,
    create_model_pipeline,
    get_feature_groups,
)

print("Centralized preprocessing utilities imported successfully.")


Centralized preprocessing utilities imported successfully.


## 3. Load Dataset

In [2]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "processed.cleveland.data"

print("Dataset path:", DATA_PATH.resolve())
print("Dataset exists:", DATA_PATH.exists())

df = load_data(DATA_PATH)
validate_columns(df, require_target=True)

print("Dataset shape:", df.shape)
display(df.head())
df.info()

print("Missing values:")
print(df.isnull().sum())


Dataset path: C:\Projects\heart-disease-prediction\data\raw\processed.cleveland.data
Dataset exists: True
Dataset shape: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


<class 'pandas.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    float64
 1   sex       303 non-null    float64
 2   cp        303 non-null    float64
 3   trestbps  303 non-null    float64
 4   chol      303 non-null    float64
 5   fbs       303 non-null    float64
 6   restecg   303 non-null    float64
 7   thalach   303 non-null    float64
 8   exang     303 non-null    float64
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    float64
 11  ca        299 non-null    float64
 12  thal      301 non-null    float64
 13  num       303 non-null    int64  
dtypes: float64(13), int64(1)
memory usage: 33.3 KB
Missing values:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
num         0
dtype: int64


## 4. Define Features and Target

In [3]:
X, y = prepare_features_target(df)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("Original target distribution:")
print(df["num"].value_counts().sort_index())


X shape: (303, 13)
y shape: (303,)
Original target distribution:
num
0    164
1     55
2     36
3     35
4     13
Name: count, dtype: int64


## 5. Convert Target to Binary Classification

In [4]:
print("Binary target distribution:")
print(y.value_counts().sort_index())

print("Binary target percentages:")
print(y.value_counts(normalize=True).sort_index() * 100)


Binary target distribution:
target
0    164
1    139
Name: count, dtype: int64
Binary target percentages:
target
0    54.125413
1    45.874587
Name: proportion, dtype: float64


### 5.1 Target and Leakage Validation

The original `num` variable is used only to construct the binary target and is not included among the predictors. No target-derived feature is created. The checks below confirm the expected target domain and predictor count before the train/test split.


In [5]:
feature_groups = get_feature_groups()
predictor_features = feature_groups["all"]
numerical_features = feature_groups["numerical"]
categorical_features = feature_groups["categorical"]

validate_columns(df, require_target=True)
assert "num" not in predictor_features
assert len(predictor_features) == 13
assert set(y.unique()) <= {0, 1}
assert len(X) == len(y) == 303

print("Predictor count:", X.shape[1])
print("Target values:", sorted(y.unique().tolist()))
print("Target leakage check: PASSED")


Predictor count: 13
Target values: [0, 1]
Target leakage check: PASSED


## 6. Identify Numerical and Categorical Features

In [6]:
print("Number of predictors:", len(predictor_features))
print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

assert set(numerical_features + categorical_features) == set(predictor_features)
print("Feature-group validation: PASSED")


Number of predictors: 13
Numerical features: 5
Categorical features: 8
Feature-group validation: PASSED


### 6.1 Missing-Value Summary

The working Cleveland file contains six missing predictor cells: four in `ca` and two in `thal`. Because the missingness mechanism cannot be established from this descriptive check alone, the project does not classify it as MCAR, MAR, or MNAR. No observations are deleted; the missing values are handled inside the training-fold preprocessing pipelines.


In [7]:
missing_summary = X.isna().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)

print("Missing predictor cells:", int(X.isna().sum().sum()))
print("Overall missing predictor-cell percentage: {:.2f}%".format(100 * X.isna().sum().sum() / (len(X) * X.shape[1])))
display(missing_summary.to_frame("Missing"))


Missing predictor cells: 6
Overall missing predictor-cell percentage: 0.15%


,Missing
ca,4
thal,2


## 7. Train/Test Split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

X_train shape: (242, 13)
X_test shape: (61, 13)
y_train shape: (242,)
y_test shape: (61,)

Training target distribution:
target
0    0.541322
1    0.458678
Name: proportion, dtype: float64

Testing target distribution:
target
0    0.540984
1    0.459016
Name: proportion, dtype: float64


## 8. Create the Shared Preprocessing Transformer

The numerical and categorical preprocessing components are implemented centrally in `src.preprocessing.create_preprocessor()`. The notebook calls that function rather than redefining `SimpleImputer`, `StandardScaler`, `OneHotEncoder`, or `ColumnTransformer` locally.


In [9]:
# The preprocessing design is centralized in src.preprocessing.create_preprocessor().
preprocessor = create_preprocessor(
    numerical_features=numerical_features,
    categorical_features=categorical_features,
)

print("Numerical preprocessing: median imputation -> StandardScaler")
print("Categorical preprocessing: most-frequent imputation -> OneHotEncoder")


Numerical preprocessing: median imputation -> StandardScaler
Categorical preprocessing: most-frequent imputation -> OneHotEncoder


## 9. Verify Categorical Preprocessing Design

Categorical preprocessing is part of the shared transformer. It uses most-frequent imputation followed by one-hot encoding with `handle_unknown="ignore"`.


In [10]:
print("Categorical preprocessing is included in the shared ColumnTransformer.")
print("OneHotEncoder uses handle_unknown='ignore' and dense output for compatibility with all baseline models.")


Categorical preprocessing is included in the shared ColumnTransformer.
OneHotEncoder uses handle_unknown='ignore' and dense output for compatibility with all baseline models.


## 10. Inspect the Shared ColumnTransformer

The transformer returned by `create_preprocessor()` combines the numerical and categorical pipelines. It remains unfitted until this notebook explicitly fits it on `X_train`.


In [11]:
print("Shared preprocessor created by src.preprocessing.create_preprocessor():")
print(preprocessor)


Shared preprocessor created by src.preprocessing.create_preprocessor():
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'trestbps', 'chol', 'thalach',
                                  'oldpeak']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['sex', 'cp', 'fbs', 'restecg', 'exang',
                                  'sl

## 11. Fit Preprocessor Using Training Data

For this verification notebook, the shared transformer is fitted only on `X_train`. In model development, `create_model_pipeline()` embeds the same transformer so that fitting occurs within each training fold during cross-validation.


In [12]:
preprocessor.fit(X_train)
print("Preprocessor fitted on training data only.")


Preprocessor fitted on training data only.


## 12. Transform Training and Testing Data

In [13]:
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_test shape:", X_test_processed.shape)

feature_names = preprocessor.get_feature_names_out()

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

display(X_train_processed_df.head())

Processed X_train shape: (242, 28)
Processed X_test shape: (61, 28)


,num__age,num__trestbps,num__chol,num__thalach,num__oldpeak,cat__sex_0.0,cat__sex_1.0,cat__cp_1.0,cat__cp_2.0,cat__cp_3.0,...,cat__slope_1.0,cat__slope_2.0,cat__slope_3.0,cat__ca_0.0,cat__ca_1.0,cat__ca_2.0,cat__ca_3.0,cat__thal_3.0,cat__thal_6.0,cat__thal_7.0
180,-0.729485,-0.395692,0.458139,0.708371,-0.445445,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
208,0.050166,-0.054513,0.230598,0.222495,-0.891627,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
167,-0.061212,0.059213,0.723605,0.399178,-0.891627,1.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
105,-0.061212,-1.305501,1.121803,0.266666,-0.891627,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
297,0.272924,0.514117,-0.167601,-1.190962,-0.713154,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


## 13. Verify Processed Dataset

### 13.1 Check Missing Values

After imputation and encoding, the transformed training and testing matrices should contain no missing values.


In [14]:
print(
    "Missing values in processed training data:",
    np.isnan(X_train_processed.toarray()).sum()
    if hasattr(X_train_processed, "toarray")
    else np.isnan(X_train_processed).sum()
)

print(
    "Missing values in processed test data:",
    np.isnan(X_test_processed.toarray()).sum()
    if hasattr(X_test_processed, "toarray")
    else np.isnan(X_test_processed).sum()
)

Missing values in processed training data: 0
Missing values in processed test data: 0


### 13.2 Get Processed Feature Names

The five numerical variables remain numerical after scaling. The eight categorical variables are one-hot encoded. For the current Cleveland data, this produces 28 transformed features.


In [15]:
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))

for i, name in enumerate(feature_names, start=1):
    print(i, name)

print("Original number of features:", X.shape[1])
print("Processed number of features:", X_train_processed.shape[1])

Number of processed features: 28
1 num__age
2 num__trestbps
3 num__chol
4 num__thalach
5 num__oldpeak
6 cat__sex_0.0
7 cat__sex_1.0
8 cat__cp_1.0
9 cat__cp_2.0
10 cat__cp_3.0
11 cat__cp_4.0
12 cat__fbs_0.0
13 cat__fbs_1.0
14 cat__restecg_0.0
15 cat__restecg_1.0
16 cat__restecg_2.0
17 cat__exang_0.0
18 cat__exang_1.0
19 cat__slope_1.0
20 cat__slope_2.0
21 cat__slope_3.0
22 cat__ca_0.0
23 cat__ca_1.0
24 cat__ca_2.0
25 cat__ca_3.0
26 cat__thal_3.0
27 cat__thal_6.0
28 cat__thal_7.0
Original number of features: 13
Processed number of features: 28


### 13.3 Convert Processed Training Data to DataFrame

In [16]:
X_train_processed_dense = (
    X_train_processed.toarray()
    if hasattr(X_train_processed, "toarray")
    else X_train_processed
)

processed_df = pd.DataFrame(
    X_train_processed_dense,
    columns=feature_names,
    index=X_train.index
)

display(processed_df.head())

,num__age,num__trestbps,num__chol,num__thalach,num__oldpeak,cat__sex_0.0,cat__sex_1.0,cat__cp_1.0,cat__cp_2.0,cat__cp_3.0,...,cat__slope_1.0,cat__slope_2.0,cat__slope_3.0,cat__ca_0.0,cat__ca_1.0,cat__ca_2.0,cat__ca_3.0,cat__thal_3.0,cat__thal_6.0,cat__thal_7.0
180,-0.729485,-0.395692,0.458139,0.708371,-0.445445,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
208,0.050166,-0.054513,0.230598,0.222495,-0.891627,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
167,-0.061212,0.059213,0.723605,0.399178,-0.891627,1.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
105,-0.061212,-1.305501,1.121803,0.266666,-0.891627,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
297,0.272924,0.514117,-0.167601,-1.190962,-0.713154,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


### 13.4 Inspect Processed Dataset

In [17]:
print("Processed dataset shape:", processed_df.shape)

display(processed_df.info())

print("Missing values in processed dataset:")
print(processed_df.isnull().sum().sum())


Processed dataset shape: (242, 28)
<class 'pandas.DataFrame'>
Index: 242 entries, 180 to 16
Data columns (total 28 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   num__age          242 non-null    float64
 1   num__trestbps     242 non-null    float64
 2   num__chol         242 non-null    float64
 3   num__thalach      242 non-null    float64
 4   num__oldpeak      242 non-null    float64
 5   cat__sex_0.0      242 non-null    float64
 6   cat__sex_1.0      242 non-null    float64
 7   cat__cp_1.0       242 non-null    float64
 8   cat__cp_2.0       242 non-null    float64
 9   cat__cp_3.0       242 non-null    float64
 10  cat__cp_4.0       242 non-null    float64
 11  cat__fbs_0.0      242 non-null    float64
 12  cat__fbs_1.0      242 non-null    float64
 13  cat__restecg_0.0  242 non-null    float64
 14  cat__restecg_1.0  242 non-null    float64
 15  cat__restecg_2.0  242 non-null    float64
 16  cat__exang_0.0    242 no

None

Missing values in processed dataset:
0


## 14. Leakage Prevention and Pipeline Check

The preprocessing transformer used for inspection is fitted on the training subset only. For model development, `create_model_pipeline()` embeds the same centralized preprocessing design inside each model pipeline. Therefore, imputation, scaling, and encoding are fitted independently within each training fold during cross-validation rather than using information from validation or test observations.


In [18]:
from sklearn.linear_model import LogisticRegression

check_pipeline = create_model_pipeline(
    LogisticRegression(max_iter=1000, random_state=42),
    numerical_features=numerical_features,
    categorical_features=categorical_features,
)

print("Pipeline integration check:", check_pipeline.named_steps)
print("Preprocessing is centralized in src.preprocessing and embedded in the model pipeline.")
print("During cross-validation, the preprocessor will be fitted independently within each training fold.")


Pipeline integration check: {'preprocessor': ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'trestbps', 'chol', 'thalach',
                                  'oldpeak']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['sex', 'cp', 'fbs', 'restecg', 'exang',
                                  'slope', 'ca', 'thal'])]), 'mo

## 15. Preprocessing Conclusions

The Cleveland heart disease dataset contains 303 observations and 13 predictors. After conversion of the original `num` variable, the binary target contains 164 observations without disease and 139 with disease. The dataset is divided into training and testing subsets using an 80:20 stratified split with `random_state = 42`, producing 242 training observations and 61 test observations.

The preprocessing strategy follows the feature types identified during EDA. The five numerical features (`age`, `trestbps`, `chol`, `thalach`, and `oldpeak`) are processed using median imputation followed by standardization. The eight categorical features (`sex`, `cp`, `fbs`, `restecg`, `exang`, `slope`, `ca`, and `thal`) are processed using most-frequent imputation followed by one-hot encoding with unknown categories ignored. The `ca` and `thal` variables are treated as categorical because their values represent discrete clinical categories rather than continuous measurements.

The working dataset contains six missing predictor cells: four in `ca` and two in `thal`, corresponding to 0.15% of all predictor cells. No rows are removed because of these values. The missingness mechanism is not inferred from this descriptive analysis.

The preprocessing transformer is fitted on training data only when this notebook demonstrates the transformation. For model development, the same transformation design is embedded directly inside each model pipeline. Consequently, imputation, scaling, and one-hot encoding are learned only from the training portion of each cross-validation fold. This prevents information from validation folds or the final test set from entering model training.

The transformed data contains 28 features: five scaled numerical features and 23 one-hot encoded categorical levels. The final test set remains untouched for model development and selection and is reserved for the final evaluation of the selected baseline model. Overall, this notebook establishes a reproducible, leakage-aware preprocessing specification for subsequent baseline and advanced modelling.
